In [17]:
import os
print(os.listdir("/content"))

['.config', 'app.py', 'data.csv', 'model.pkl', '__pycache__', 'sample_data']


In [18]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Load dataset
df = pd.read_csv("/content/data.csv")

# Remove unnecessary columns
df = df.drop(["id", "Unnamed: 32"], axis=1)

# Convert diagnosis into numbers
df["diagnosis"] = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

# Separate input and output
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create model pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train model
model.fit(X_train, y_train)

# Test model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model trained successfully!")
print("Accuracy:", accuracy)

# Save model
joblib.dump(model, "/content/model.pkl")

print("model.pkl created successfully!")

Model trained successfully!
Accuracy: 0.9736842105263158
model.pkl created successfully!


In [19]:
import os

print(os.listdir("/content"))

['.config', 'app.py', 'data.csv', 'model.pkl', '__pycache__', 'sample_data']


In [20]:
print(os.path.exists("/content/model.pkl"))

True


In [21]:
%%writefile /content/app.py

from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)

# Load trained model
model = joblib.load("/content/model.pkl")

@app.route("/")
def home():
    return "ML Model API is running!"

@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    features = data["features"]

    prediction = model.predict([features])[0]

    if prediction == 1:
        result = "Malignant"
    else:
        result = "Benign"

    return jsonify({
        "prediction": int(prediction),
        "result": result
    })

Overwriting /content/app.py


In [22]:
import os

print(os.listdir("/content"))

['.config', 'app.py', 'data.csv', 'model.pkl', '__pycache__', 'sample_data']


In [23]:
!pip install flask pyngrok joblib

In [24]:
import threading
import time
from app import app

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

time.sleep(3)

print("Flask server started!")

 * Serving Flask app 'app'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Flask server started!


In [25]:
import requests

response = requests.get("http://127.0.0.1:5000/")

print(response.status_code)
print(response.text)

INFO:werkzeug:127.0.0.1 - - [03/Sep/2026 15:44:53] "GET / HTTP/1.1" 200 -


200
ML Model API is running!


In [26]:
import requests

data = {
    "features": [
        17.99, 10.38, 122.8, 1001.0, 0.1184,
        0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
        1.095, 0.9053, 8.589, 153.4, 0.006399,
        0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.6, 2019.0, 0.1622,
        0.6656, 0.7119, 0.2654, 0.4601, 0.1189
    ]
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=data
)

print(response.status_code)
print(response.json())

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
INFO:werkzeug:127.0.0.1 - - [03/Sep/2026 15:44:53] "POST /predict HTTP/1.1" 200 -


200
{'prediction': 1, 'result': 'Malignant'}
